# Fine-tuning a Large Language Model

In this lecture we will be looking at how to fine-tune an existing pre-trained language model.

## Learning outcomes
* You will learn how to download a pre-trained model and a training dataset from Hugging Face.
* You will learn how to fine-tune the downloaded model with the dataset using Hugging Face trl library and the supervised fine-tuning (SFT) method.
* You will learn how to use the fine-tuned model to generate text based on user input / prompts.
* You will learn how to upload the fine-tuned model to your own Hugging Face repository so that it can be used later or shared with other users.

## Prerequistes
* You will need the following free accounts: Google, Hugging Face and Weights & Biases. You may use your existing accounts or create new accounts for the purposes of this course.
* We will use the [Hugging Face](https://huggingface.co/) libraries: transformers (for models), datasets (for datasets), trl (for training). We will also store the fine-tuned models in a Hugging Face repository.
* Training is done using [Google Colab](https://colab.research.google.com/), which provides free access to Jupyter notebooks backed with a GPU compute required for fine-tuning.
* For monitoring the training run we will use [Weights & Biases](https://wandb.ai/)


## Fine-tuning

Let's first install some pre-requisites using Python's package manager pip

In [ ]:
!pip install transformers peft accelerate
!pip install -U datasets
!pip install -q trl==0.12.0 xformers wandb einops sentencepiece bitsandbytes
!pip install --upgrade huggingface_hub

Then we need to import the required libraries

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, TextStreamer
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
import torch, wandb
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from huggingface_hub import notebook_login

We will download a pre-trained large language model from Hugging Face and a dataset to train the model with. Below we assign these to variables we will use later. We will also set the name of the repository and model for the fine-tuned model.

In [ ]:
# Pre trained model
model_name = "Qwen/Qwen2.5-7B-Instruct"

# Dataset name
dataset_name = "lucasmccabe-lmi/gpt4all_code"

# Hugging face repository link to save fine-tuned model(Create new repository in huggingface,copy and paste here)
new_model = "heleneil/qwen2.5-7B-finetune"

To access your Hugging Face account, you need to log in. First go to your Hugging Face account, click *Settings* and select *Access Tokens*. Create a new token and copy the token. Then execute the below login command and when asked paste an access token.  

In [ ]:
notebook_login()

Let's then download a subset of the dataset we want to use. Below we limit the dataset to the first 10,000 examples in order to save time. In real life you would probably use the full dataset.

In [ ]:
dataset = load_dataset(dataset_name, split="train[0:100]")
print(dataset[0]["instruction"], dataset[0]["output"])

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


<p>I am building a website with Wordpress and have a question regarding my homepage. It is divided into 4 container divs, each of them taking up the entire screen. In my 4th div, I want to have something like a tab-function. Meaning, there are 4 buttons at the bottom of the container4 div and depending on which of the four buttons you click, a different content will load (content 1, 2, 3 or 4).</p>

<p>I found a useful code on codepen and altered it a bit for my needs but I am encountering an issues I can't seem to solve.</p>

<p>The buttons are wrapped in a 'button-wrap' div and then I also have the content div that encloses the 4 divs with the content that's supposed to be swapped out when clicking on the buttons. This works great with the jQuery code I have and the content divs are also all nicely lining up with the main Container 4 div. However, even though I did place the button wrap div in the container div as well, it is being pushed down into the footer of my homepage. I tried 

Let's then download the model. We first create a config object for quantization of the model using bitsandbytes. Bitsandbytes enables accessible large language models via k-bit quantization for PyTorch.

We also need to download the tokenizer.

In [ ]:
def formatting_func(example):
    text = f"<|instruction|>\n{example['instruction']}\n<|output|>\n{example['output']}"
    return {"text": text}

print(formatting_func(dataset[0]))

{'text': '<|instruction|>\n<p>I am building a website with Wordpress and have a question regarding my homepage. It is divided into 4 container divs, each of them taking up the entire screen. In my 4th div, I want to have something like a tab-function. Meaning, there are 4 buttons at the bottom of the container4 div and depending on which of the four buttons you click, a different content will load (content 1, 2, 3 or 4).</p>\n\n<p>I found a useful code on codepen and altered it a bit for my needs but I am encountering an issues I can\'t seem to solve.</p>\n\n<p>The buttons are wrapped in a \'button-wrap\' div and then I also have the content div that encloses the 4 divs with the content that\'s supposed to be swapped out when clicking on the buttons. This works great with the jQuery code I have and the content divs are also all nicely lining up with the main Container 4 div. However, even though I did place the button wrap div in the container div as well, it is being pushed down into 

In [ ]:
formatted_dataset = dataset.map(formatting_func)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit= True,
    bnb_4bit_quant_type= "nf4",
    bnb_4bit_compute_dtype= torch.float16,
    bnb_4bit_use_double_quant= False,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0}
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False # silence the warnings. Please re-enable for inference!
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_eos_token = True
tokenizer.add_eos_token

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

True

In [ ]:
print(tokenizer.special_tokens_map)

{'eos_token': '<|im_end|>', 'pad_token': '<|im_end|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}


Below we set the access token to Waights & Biases. You should copy your access token from your account at [https://wandb.ai](https://wandb.ai).

In [ ]:
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding=True, max_length=512)

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)

In [ ]:
#monitering login
wandb.login(key="") # Add your WANDB key here
run = wandb.init(project='Fine tuning Qwen 2.5-7B', job_type="training", anonymous="allow")

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: heleneil (heleneil-university-of-helsinki). Use `wandb login --relogin` to force relogin


Then we'll create a configuration for the lo-rank adaptation method we will use.

In [ ]:
peft_config = LoraConfig(
    lora_alpha=8,
    lora_dropout=0.1,
    r=16,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj"]
)

We need to set the training arguments for the training run.

In [ ]:
training_arguments = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    optim="paged_adamw_8bit",
    save_steps=1000,
    logging_steps=30,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.3,
    group_by_length=True,
    lr_scheduler_type="linear",
    report_to="wandb",
    dataset_text_field="text",
    max_seq_length= None,
    packing=False,
)

Finally we create the trainer object that uses supervised fine-tuning (SFT) as the training method.

In [ ]:
# Setting sft parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=training_arguments,
    formatting_func=formatting_func,
)

/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:309: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:463: UserWarning: You passed a dataset that is already processed (contains an `input_ids` field) together with a valid formatting function. Therefore `formatting_func` will be ignored.
  warnings.warn(


Then, we can execute the training run. This will approximately 8 hours using the T4 GPU available in Colab and the dataset of 10,000 samples we downloaded.

In [ ]:
# Train model
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss


TrainOutput(global_step=12, training_loss=1.3293337027231853, metrics={'train_runtime': 626.13, 'train_samples_per_second': 0.16, 'train_steps_per_second': 0.019, 'total_flos': 2094139667644416.0, 'train_loss': 1.3293337027231853, 'epoch': 0.96})

In [ ]:
# Save the fine-tuned model
trainer.model.save_pretrained(new_model)
wandb.finish()
model.config.use_cache = True
model.eval()

train/epoch,▁
train/global_step,▁
total_flos,2094139667644416.0
train/epoch,0.96
train/global_step,12
train_loss,1.32933
train_runtime,626.13
train_samples_per_second,0.16
train_steps_per_second,0.019


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.1, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=3584, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=3584, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=3584, out_features=512, bias=True)
            (lora_dropout): ModuleDict(
      

In [ ]:
def stream(user_prompt):
    runtimeFlag = "cuda:0"

    B_INST, E_INST = "<|instruction|>", "<|response|>"

    prompt = f"{B_INST} {user_prompt.strip()} {E_INST}"

    inputs = tokenizer([prompt], return_tensors="pt").to(runtimeFlag)

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    output_ids = model.generate(**inputs, max_new_tokens=500)

    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print(output_text)
    return output_text

In [ ]:
stream("What is the difference between == and is in Python?")

<|instruction|> What is the difference between == and is in Python? <|response|> In Python, `==` and `is` are both comparison operators, but they have different meanings.

1. `==`: This operator checks if the values of two operands are equal or not. If the values are equal, then the condition becomes `True`. It compares the contents of the objects.

Example:
```python
a = [1, 2, 3]
b = [1, 2, 3]
c = a

print(a == b)  # Output: False
print(a == c)  # Output: True
```
In the first example, `a` and `b` have the same elements but are stored in different memory locations, so `a == b` returns `False`. In the second example, `a` and `c` refer to the same object (the list), so `a == c` returns `True`.

1. `is`: This operator checks whether two variables refer to the same object or not. It compares the memory addresses of the objects. If both variables refer to the same object, then the condition becomes `True`.

Example:
```python
a = [1, 2, 3]
b = [1, 2, 3]
c = a

print(a is b)  # Output: Fal

'<|instruction|> What is the difference between == and is in Python? <|response|> In Python, `==` and `is` are both comparison operators, but they have different meanings.\n\n1. `==`: This operator checks if the values of two operands are equal or not. If the values are equal, then the condition becomes `True`. It compares the contents of the objects.\n\nExample:\n```python\na = [1, 2, 3]\nb = [1, 2, 3]\nc = a\n\nprint(a == b)  # Output: False\nprint(a == c)  # Output: True\n```\nIn the first example, `a` and `b` have the same elements but are stored in different memory locations, so `a == b` returns `False`. In the second example, `a` and `c` refer to the same object (the list), so `a == c` returns `True`.\n\n1. `is`: This operator checks whether two variables refer to the same object or not. It compares the memory addresses of the objects. If both variables refer to the same object, then the condition becomes `True`.\n\nExample:\n```python\na = [1, 2, 3]\nb = [1, 2, 3]\nc = a\n\nprin

<|instruction|> What is the difference between == and is in Python? <|response|> In Python, `==` and `is` are two operators used for comparison, but they serve different purposes.

1. `==` (Equality operator): This operator checks if the values of two operands are equal or not. It returns `True` if the values are equal and `False` otherwise. For example:

```python
a = 5
b = 5
print(a == b)  # Output: True

c = [1, 2, 3]
d = [1, 2, 3]
print(c == d)  # Output: True
```

2. `is` (Identity operator): This operator checks if both the operands refer to the same object in memory. It returns `True` if the operands refer to the same object and `False` otherwise. For example:

```python
a = 5
b = 5
print(a is b)  # Output: False

c = [1, 2, 3]
d = c
print(c is d)  # Output: True
```

In the second example, `c` and `d` refer to the same list object in memory, so `c is d` returns `True`. However, when comparing two integers with `is`, it might return `False` because Python may create new integer objects for different values to save memory.

In summary, use `==` when you want to check if the values of two operands are equal, and use `is` when you want to check if the operands refer to the same object in memory. >

In [ ]:
# This will fail due to cuda out of memeory issue. Need to add quantization
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    device_map= {"": 0})
model = PeftModel.from_pretrained(base_model, new_model)
model = model.merge_and_unload()

# Reload tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/peft/tuners/lora/bnb.py:355: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [ ]:
model.push_to_hub(repo_id=new_model)
tokenizer.push_to_hub(repo_id=new_model)

model-00002-of-00002.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.76G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/heleneil/qwen2.5-7B-finetune/commit/05d57f7ae1121a4baae0cd283df2ad37dba89e54', commit_message='Upload tokenizer', commit_description='', oid='05d57f7ae1121a4baae0cd283df2ad37dba89e54', pr_url=None, repo_url=RepoUrl('https://huggingface.co/heleneil/qwen2.5-7B-finetune', endpoint='https://huggingface.co', repo_type='model', repo_id='heleneil/qwen2.5-7B-finetune'), pr_revision=None, pr_num=None)